In [2]:
# all imports

# pandas, geopandas, numpy, os imports
import geopandas as gpd
import pandas as pd
import numpy as np
import os

# matplotlib imports
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from matplotlib.collections import PatchCollection

# shapely imports
from shapely.geometry import Polygon, MultiPolygon, LineString, Point
#from shapely.errors import ShapelyError
#from shapely.ops import unary_union

In [3]:
# get working directory
current_dir = os.getcwd()
print(current_dir)

# change wd
os.chdir(r"C:\Users\BCulligan\Desktop\cropfield-polygonization")

c:\Users\BCulligan\Desktop\Week5\cropfield-polygonization\images


In [4]:
# load data
df = pd.read_parquet(r"zambia_2023_attributed.parquet")
gdf = gpd.read_parquet(r'zambia_2023_attributed.parquet')

# view data in table format
df.head(5)

,area_m2,area_ha,perimeter_m,compactness,shape_index,interior_edge,fractal_dim,tile_id,polygon_index,polygon_id,geometry
0,151068.586899,15.106859,4141.286922,0.110691,3.005684,0.027413,1.396800,895920.0,0.0,8.959200e+11,b'\x01\x03\x00\x00\x00\x0c\x00\x00\x00\xaa\x00...
1,22061.759097,2.206176,608.624031,0.748431,1.155910,0.027587,1.282035,895920.0,1.0,8.959200e+11,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x10\x00..."
2,1307.204778,0.130720,154.951184,0.684169,1.208977,0.118536,1.405618,895920.0,2.0,8.959200e+11,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x06\x00...
3,3958.970804,0.395897,299.582238,0.554319,1.343136,0.075672,1.376767,895920.0,3.0,8.959200e+11,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x10\x00...
4,683.484230,0.068348,97.299749,0.907225,1.049887,0.142358,1.402682,895920.0,4.0,8.959200e+11,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x08\x00..."


In [5]:
# choose one polygon to work with

poly = gdf[gdf['polygon_id'] == 862215000279]  # based on polygon_id

# verify that just one polygon is selected
print(poly.count())

# view polygon table
print(poly.head(5))

# extract coordinates of the selected polygon

poly_coords = gdf.iloc[0]['geometry']
print(poly_coords)

area_m2          1
area_ha          1
perimeter_m      1
compactness      1
shape_index      1
interior_edge    1
fractal_dim      1
tile_id          1
polygon_index    1
polygon_id       1
geometry         1
dtype: int64
              area_m2   area_ha  perimeter_m  compactness  shape_index  \
1015114  46183.780882  4.618378  2163.212455     0.124023      2.83955   

         interior_edge  fractal_dim   tile_id  polygon_index    polygon_id  \
1015114       0.046839     1.429995  862215.0          279.0  8.622150e+11   

                                                  geometry  
1015114  POLYGON ((30.18202 -11.00056, 30.18197 -11.000...  
POLYGON ((23.412389194597296 -14.25763631815908, 23.4124392196098 -14.25763631815908, 23.412589294647322 -14.257536268134068, 23.412714357178587 -14.257261130565283, 23.41298949474737 -14.257286143071536, 23.413114557278636 -14.257211105552777, 23.41321460730365 -14.257211105552777, 23.413314657328662 -14.257261130565283, 23.413314657328662 -14.257

In [6]:
# separate shell from holes, get coordinates

# shell
shell = poly_coords.exterior
print(f"Polygon Shell: {list(shell.coords)}") # one object

# holes
holes = poly_coords.interiors
print(f"\nPolygon Holes: {holes}") # many objects, therefore list is inappropriate. \n = new line
for i, hole in enumerate(holes):
    print(f"Hole {i}:") # separates holes and labels them according to their index
    print(list(hole.coords)) # since this is within an interation, coordinates can be printed for each hole


Polygon Shell: [(23.412389194597296, -14.25763631815908), (23.4124392196098, -14.25763631815908), (23.412589294647322, -14.257536268134068), (23.412714357178587, -14.257261130565283), (23.41298949474737, -14.257286143071536), (23.413114557278636, -14.257211105552777), (23.41321460730365, -14.257211105552777), (23.413314657328662, -14.257261130565283), (23.413314657328662, -14.257311155577788), (23.41346473236618, -14.257411205602802), (23.41346473236618, -14.257686343171585), (23.41328964482241, -14.257986493246623), (23.413239619809904, -14.258186593296648), (23.41328964482241, -14.259312156078039), (23.41338969484742, -14.259437218609305), (23.413439719859927, -14.259362181090545), (23.413589794897447, -14.259312156078039), (23.4136148074037, -14.25916208104052), (23.413514757378685, -14.259287143571786), (23.413364682341168, -14.259287143571786), (23.413364682341168, -14.258886943471737), (23.413489744872432, -14.258861930965482), (23.413564782391195, -14.258886943471737), (23.41363

In [7]:
# check if polygon has holes

if len(poly_coords.interiors) > 0: # checks if there are more than 0 holes. len is like a total
    print(f"Number of holes: {len(poly_coords.interiors)}") # f allows for different formatting
    
    # shell points within polygon
    shell_coords = list(poly_coords.exterior.coords) # line -> list of coordinates
    print(f"Shell has {len(shell_coords)} points")
    
    # points within polygon for holes
    for i, hole in enumerate(poly_coords.interiors):
        hole_coords = list(hole.coords)
        print(f"Hole {i} has {len(hole_coords)} points")
else:
    print("This polygon has no holes (no interior boundaries)")

Number of holes: 11
Shell has 170 points
Hole 0 has 5 points
Hole 1 has 4 points
Hole 2 has 8 points
Hole 3 has 5 points
Hole 4 has 4 points
Hole 5 has 4 points
Hole 6 has 5 points
Hole 7 has 6 points
Hole 8 has 5 points
Hole 9 has 8 points
Hole 10 has 4 points


In [8]:
# define function to remove holes from a polygon

def remove_holes(geom, should_remove): # geom = polygon's geometry and should_remove = built in function
    def p(p: Polygon, should_remove) -> Polygon:
        holes = [i for i in p.interiors if not should_remove(Polygon(i))]
        return Polygon(shell=p.exterior, holes=holes)

    def mp(mp: MultiPolygon, should_remove) -> MultiPolygon:
        return MultiPolygon([p(i, should_remove) for i in mp.geoms])

    if isinstance(geom, Polygon):
        return p(geom, should_remove)
    elif isinstance(geom, MultiPolygon):
        return mp(geom, should_remove)
    else:
        return geom

In [9]:
cleaned_poly_coords = remove_holes(poly_coords, should_remove = lambda x: True) # this should_remove function removes all holes
print(cleaned_poly_coords)

POLYGON ((23.412389194597296 -14.25763631815908, 23.4124392196098 -14.25763631815908, 23.412589294647322 -14.257536268134068, 23.412714357178587 -14.257261130565283, 23.41298949474737 -14.257286143071536, 23.413114557278636 -14.257211105552777, 23.41321460730365 -14.257211105552777, 23.413314657328662 -14.257261130565283, 23.413314657328662 -14.257311155577788, 23.41346473236618 -14.257411205602802, 23.41346473236618 -14.257686343171585, 23.41328964482241 -14.257986493246623, 23.413239619809904 -14.258186593296648, 23.41328964482241 -14.259312156078039, 23.41338969484742 -14.259437218609305, 23.413439719859927 -14.259362181090545, 23.413589794897447 -14.259312156078039, 23.4136148074037 -14.25916208104052, 23.413514757378685 -14.259287143571786, 23.413364682341168 -14.259287143571786, 23.413364682341168 -14.258886943471737, 23.413489744872432 -14.258861930965482, 23.413564782391195 -14.258886943471737, 23.413639819909953 -14.259012006003001, 23.41368984492246 -14.258986993496748, 23.41

In [10]:
# check if polygon has holes

if len(cleaned_poly_coords.interiors) > 0: # checks if there are more than 0 holes. len is like a total
    print(f"Number of holes: {len(cleaned_poly_coords.interiors)}") # f allows for different formatting

    # shell points within polygon
    shell_coords = list(cleaned_poly_coords.exterior.coords) # line -> list of coordinates
    print(f"Shell has {len(shell_coords)} points")
    
    # points within polygon for holes
    for i, hole in enumerate(cleaned_poly_coords.interiors):
        hole_coords = list(hole.coords)
        print(f"Hole {i} has {len(hole_coords)} points")
else:
    print("This polygon has no holes. Successfully removed all holes from the original polygon!")

This polygon has no holes. Successfully removed all holes from the original polygon!
